# 02 — Anchor System Comparison

This notebook compares three common anchor configurations across a full range of load angles:

- **Sliding-X** (Magic X / Italian hitch equalising)
- **Quad** (redundant pre-equalised)
- **Cordelette** (fixed distribution)

We sweep the load angle from −90° to +90° and plot force on each bolt.

In [ ]:
import ropesim.notebook
from ropesim import AnchorSystem, AnchorType, Bolt, BoltType, RockType
import numpy as np
import matplotlib.pyplot as plt

## Build the three anchor configurations

Each uses two bolts with 25 kN MBS in granite — a typical sport-climbing anchor.

In [ ]:
def make_anchor(anchor_type):
    b1 = Bolt(position=[0.0, 0.0], mbs_kn=25.0, bolt_type=BoltType.GLUE_IN,
              rock_type=RockType.GRANITE)
    b2 = Bolt(position=[0.4, 0.0], mbs_kn=25.0, bolt_type=BoltType.GLUE_IN,
              rock_type=RockType.GRANITE)
    return AnchorSystem(components=[b1, b2], anchor_type=anchor_type)

anchors = {
    'Sliding-X':   make_anchor(AnchorType.SLIDING_X),
    'Quad':        make_anchor(AnchorType.QUAD),
    'Cordelette':  make_anchor(AnchorType.CORDELETTE),
}
print('Anchors built:', list(anchors.keys()))

## Display one anchor card

In [ ]:
anchors['Quad']  # rich HTML card

## Angle sweep

We load each anchor at 10 kN across angles from −90° (hard left) to +90° (hard right) and record per-bolt forces.

In [ ]:
angles_deg = np.linspace(-90, 90, 181)
results = {name: {'left': [], 'right': []} for name in anchors}

for name, anchor in anchors.items():
    for angle in angles_deg:
        forces = anchor.compute_bolt_forces(load_kn=10.0, load_angle_deg=angle)
        results[name]['left'].append(forces[0])
        results[name]['right'].append(forces[1])

## Plot per-bolt forces across angles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
colours = {'left': '#2270c4', 'right': '#e74c3c'}

for ax, (name, data) in zip(axes, results.items()):
    ax.plot(angles_deg, data['left'],  color=colours['left'],  label='Left bolt',  linewidth=1.8)
    ax.plot(angles_deg, data['right'], color=colours['right'], label='Right bolt', linewidth=1.8)
    ax.axhline(10.0, color='grey', linestyle=':', linewidth=0.8, label='Applied load')
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.5)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Load angle (°)')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 18)

axes[0].set_ylabel('Bolt force (kN)')
fig.suptitle('Per-bolt forces vs load angle — 10 kN applied load', fontsize=13)
plt.tight_layout()
plt.show()

## Worst-case bolt force heatmap

Sweep both angle (−90° to +90°) and load magnitude (0–15 kN). For each combination, record the maximum force on any single bolt.

In [ ]:
loads_kn = np.linspace(1, 15, 50)
angle_grid = np.linspace(-90, 90, 91)

heatmaps = {}
for name, anchor in anchors.items():
    heat = np.zeros((len(loads_kn), len(angle_grid)))
    for i, load in enumerate(loads_kn):
        for j, angle in enumerate(angle_grid):
            forces = anchor.compute_bolt_forces(load_kn=load, load_angle_deg=angle)
            heat[i, j] = max(forces)
    heatmaps[name] = heat

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, heat) in zip(axes, heatmaps.items()):
    im = ax.pcolormesh(angle_grid, loads_kn, heat, cmap='RdYlGn_r', vmin=0, vmax=18)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Load angle (deg)')
    ax.set_ylabel('Applied load (kN)' if ax == axes[0] else '')
    fig.colorbar(im, ax=ax, label='Max bolt force (kN)')

fig.suptitle('Worst-case bolt force: load x angle heatmap', fontsize=13)
plt.tight_layout()
plt.show()

## Symmetric load distribution comparison

At 0° (perfectly centred load), how equally is force distributed between the two bolts?

In [ ]:
print('Bolt force distribution at 0 deg load angle, 10 kN applied:')
print(f'{"Anchor":<15}  {"Left bolt":>10}  {"Right bolt":>10}  {"Imbalance":>10}')
for name, anchor in anchors.items():
    f = anchor.compute_bolt_forces(load_kn=10.0, load_angle_deg=0.0)
    imbalance = abs(f[0] - f[1])
    print(f'{name:<15}  {f[0]:>9.2f}kN  {f[1]:>9.2f}kN  {imbalance:>9.2f}kN')

## Key takeaways

- The **Quad** is notably more stable across angles — per-bolt forces vary less than the Sliding-X.
- The **Cordelette** redistributes poorly for off-axis loads (one bolt gets almost the full force at large angles).
- **Sliding-X** naturally equalises with the load direction but can produce a shock-load if one bolt fails (no extension limiter).
- For sport anchors, EN 959 allows 25 kN single-bolt MBS — a well-placed two-bolt anchor has enormous safety margin at typical 8–12 kN fall forces.